# Applied AI Assignment 2 — Cross-Domain Sentiment Classification

Dataset: Multi-Domain Sentiment Dataset, version 2.0. Task: train on Books and evaluate on DVD, Electronics and Kitchen. The notebook runs exactly two traditional models and two deep models, with a baseline and refined version of each.

The dataset is downloaded from the official Johns Hopkins source. Raw data are not saved in the repository.

In [ ]:
from pathlib import Path
import sys, subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from assignment1.data import load_multidomain_dataset, dataset_summary
from assignment1.experiment import run_experiment
DATA_DIR = ROOT / 'data' / 'raw' / 'processed_acl'
RESULTS_DIR = ROOT / 'results'
SEED = 42

## 1. Dataset download and inspection

The official version 2.0 page provides a balanced binary archive in the processed ACL format. Each record stores word-feature counts followed by a sentiment label.

In [ ]:
if not DATA_DIR.exists() or not any(DATA_DIR.rglob('processed.review.balanced')):
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'download_dataset.py')], check=True)
dataset = load_multidomain_dataset(DATA_DIR)
full_summary = pd.DataFrame(dataset_summary(dataset))
display(full_summary)

## 2. Data analysis

The experiment uses a reproducible, class-balanced sample of up to 1,500 positive and 1,500 negative reviews per domain. The source split is stratified. Target domains are kept out of model fitting.

In [ ]:
sample_counts = full_summary[['domain', 'positive', 'negative', 'documents']].copy()
display(sample_counts)
ax = full_summary.set_index('domain')[['positive', 'negative']].plot(kind='bar', figsize=(8,4))
ax.set_title('Full dataset class counts by domain')
ax.set_xlabel('Domain'); ax.set_ylabel('Reviews'); ax.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.show()

## 3. Model training and refinement

Traditional models: Logistic Regression and Linear SVM. Deep models: DNN and Residual MLP. Traditional refinements tune regularisation on validation F1. Neural refinements change the hidden architecture and regularisation, then use early stopping.

In [ ]:
metrics = run_experiment(
    data_dir=DATA_DIR,
    output_dir=RESULTS_DIR,
    source_domain='books',
    target_domains=['dvd', 'electronics', 'kitchen'],
    max_per_class=1500,
    max_features=5000,
    seed=SEED,
)
display(metrics.sort_values(['evaluation', 'model', 'stage']))

## 4. Results and evaluation

The evaluation reports accuracy, precision, recall, F1-score and ROC-AUC. The same table is written to `results/metrics_summary.csv` and inserted into the LaTeX report.

In [ ]:
refinement = pd.read_csv(RESULTS_DIR / 'refinement_comparison.csv')
display(refinement)
print('Unified model evaluation table:')
display(pd.read_csv(RESULTS_DIR / 'metrics_summary.csv'))

## 5. Visualisations

Logistic Regression estimates a positive-class probability using the sigmoid function:
$$P(y=1|x)=\\frac{1}{1+e^{-(w^Tx+b)}}$$
The F1-score is the harmonic mean of precision and recall:
$$F1=2\\frac{Precision\\times Recall}{Precision+Recall}$$

In [ ]:
for filename in ['domain_class_balance.png', 'top_source_features.png', 'model_adaptation_gap.png']:
    print(filename)
    display(Image(filename=str(RESULTS_DIR / 'figures' / filename)))

## 6. Saved outputs

The experiment creates the dataset summaries, unified evaluation metrics, refinement log, model comparison figures and confusion matrices. The report build step reads these outputs and inserts the measured results. No metric is manually entered.

In [ ]:
print('Output files:')
for path in sorted(RESULTS_DIR.rglob('*')):
    if path.is_file(): print(path.relative_to(ROOT))